<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [ ]:
!pip3 install folium
!pip3 install wget
!pip3 install pandas

In [ ]:
import folium
import wget
import pandas as pd

In [ ]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [ ]:
# Download and read the `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [ ]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [ ]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [ ]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [ ]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


- Are all launch sites in proximity to the Equator line?
  
  No. While the launch sites (such as Cape Canaveral / Kennedy Space Center in Florida around $28.5^\circ\text{N}$ and Vandenberg Space Force Base in California around $34.6^\circ\text{N}$) are situated at relatively low-to-mid latitudes compared to high-latitude or polar regions, they are not directly in proximity to the Equator.


- Are all launch sites in very close proximity to the coast?

  Yes. All primary launch locations are positioned directly adjacent to the coastline, opening up wide ocean ranges downrange.

  Explanation of Findings
Range Safety: Placing launch pads on the coast ensures that rockets fly eastward (or southward for polar orbits) over unpopulated ocean waters. If a vehicle experiences an anomaly or a flight termination system is triggered, falling debris lands safely in the sea rather than on populated communities.

Earth's Rotational Velocity: Earth rotates toward the east, generating maximum rotational speed at the Equator (~1,670 km/h). By positioning launch facilities at lower latitudes (such as Florida), rockets inherit a portion of this rotational momentum, which acts as a free velocity boost toward reaching orbital speed and increases payload capacity.

Trajectory and Inclination Flexibility: Coastal sites allow for diverse launch azimuths without crossing landmasses. For instance, polar and sun-synchronous orbits are ideally launched from West Coast facilities (like Vandenberg) directly over the Pacific Ocean to achieve high-inclination paths safely.



# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [ ]:
spacex_df.tail(10)

Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [ ]:
marker_cluster = MarkerCluster()


_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [ ]:
import pandas as pd

# Define a function to assign marker color based on class
def assign_marker_color(launch_class):
    if launch_class == 1:
        return 'green'
    else:
        return 'red'

# Apply the function to create the new 'marker_color' column
# (Assuming your dataframe is named 'data_falcon9' or 'launch_sites')
data_falcon9['marker_color'] = data_falcon9['class'].apply(assign_marker_color)

# Display the first few rows to verify the new column
data_falcon9[['FlightNumber', 'LaunchSite', 'class', 'marker_color']].head()

In [ ]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [ ]:
# Assuming 'marker_cluster' is already initialized and added to your map
# and 'spacex_df' contains the columns: 'Lat', 'Long', and 'marker_color'

for index, row in spacex_df.iterrows():
    # Define the coordinates
    marker_location = [row['Lat'], row['Long']]
    
    # Create the marker with the assigned color
    marker = folium.Marker(
        location=marker_location,
        icon=folium.Icon(color='white', icon_color=row['marker_color']),
        popup=f"Launch Site: {row['LaunchSite']}"
    )
    
    # Add the marker to the marker cluster
    marker_cluster.add_child(marker)

# Display the final map with the clusters
site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [ ]:
from folium.plugins import MousePosition

# Add MousePosition to the map
# This displays coordinates in the top-right corner when you hover over the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"

mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)

# Display the map to start exploring
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [ ]:
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

# 1. Define your coordinates
# Replace these with the specific coordinates you found
launch_site_coords = [28.5721, -80.6480] # Example: KSC LC-39A
coastline_coords = [28.5721, -80.5700]   # Example: Nearest coastline point

# 2. Calculate the distance
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0 # radius of the earth
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

distance = calculate_distance(launch_site_coords[0], launch_site_coords[1], coastline_coords[0], coastline_coords[1])


_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [ ]:

# 3. Create and add the Distance Marker
distance_marker = folium.Marker(
    coastline_coords,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
    )
)
site_map.add_child(distance_marker)



_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [ ]:
Create and add the Distance Marker
distance_marker = folium.Marker(
    coastline_coords,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
    )
)
site_map.add_child(distance_marker)


_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [ ]:
# Create a list of coordinates [ [launch_lat, launch_lon], [coast_lat, coast_lon] ]
coordinates = [launch_site_coords, coastline_coords]
lines = folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [ ]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site


In [ ]:
from folium.features import DivIcon

# 1. Define the coordinates for the Point of Interest (POI)
# Replace these values with the coordinates you found using the MousePosition tool
poi_coords = [28.5236, -80.6425]  # Example: Coordinates for a nearby Railway or City
label_name = "City"               # Label for your marker

# 2. Calculate distance
distance = calculate_distance(launch_site_coords[0], launch_site_coords[1], poi_coords[0], poi_coords[1])

# 3. Create the Marker with distance label
marker = folium.Marker(
    poi_coords,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s: %10.2f KM</b></div>' % (label_name, distance),
    )
)
site_map.add_child(marker)

# 4. Draw the PolyLine
lines = folium.PolyLine(locations=[launch_site_coords, poi_coords], weight=1)
site_map.add_child(lines)

# Display map
site_map

After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.

Based on the visual analysis performed with your map and the proximity calculations, here are the answers to your questions regarding the location strategy of SpaceX launch sites:

Analysis of Launch Site Proximities
Are launch sites in close proximity to railways?

Yes. You will likely find that launch sites are located within a few kilometers of major railway spurs.

Are launch sites in close proximity to highways?

Yes. Launch sites are consistently positioned near major highways or access roads that connect to the broader national road network.

Are launch sites in close proximity to coastline?

Yes. This is the most consistent feature; all primary launch pads are situated directly adjacent to the coast.

Do launch sites keep a certain distance away from cities?

Yes. While they require infrastructure (highways/rail), they maintain a significant "buffer zone" of several kilometers or more from densely populated urban centers.

Explanation of Findings
The location of these launch sites is not random; it is the result of strict aerospace logistics and safety requirements:

Logistics & Heavy Transport (Rail & Highway Proximity):
Rocket hardware is massive. Stages, specialized support equipment, and hazardous propellant tanks are often too large or heavy to be moved by standard road transport alone. Railways provide the most efficient way to transport heavy rocket stages from manufacturing facilities to the integration hangars at the launch site. Highways are equally critical for the day-to-day transport of personnel, supplies, and smaller components.

Range Safety (Coastline Proximity):
Launching over the ocean is a requirement for modern spaceflight. In the event of a vehicle failure shortly after liftoff, you need a "downrange" area that is unpopulated. Placing launch sites on the coast allows rockets to fly trajectories over the open ocean, ensuring that debris from an unexpected failure lands in the water rather than on homes or businesses.

Safety Buffers & Zoning (Distance from Cities):
Launch sites are inherently hazardous due to the massive quantities of explosive propellants, the extreme acoustic energy (noise) generated during ignition, and the potential for chemical hazards. A significant distance from cities serves as a critical safety buffer, protecting residents from:

Acoustic Damage: The shockwaves from a rocket launch can be powerful enough to shatter windows and cause structural damage to nearby buildings.

Hazardous Materials: Handling fuels (like RP-1, Liquid Oxygen, or Methane) carries risks of leaks or fires.

Emergency Evacuation: If an accident occurs, maintaining distance ensures there is enough space to manage emergency operations without requiring the immediate evacuation of a metropolitan population.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
